# Overview

In this notebook we do the follow in order:
- Read human and AI corpus
- Extract numerical features
- Create training and testing sets X and Y
- Fit scikit-learn model
- Evaluate on test set
- Analyze individual features (E.g. "are longer senteces more 'AI-like' to our model?")

In [79]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

# Data prep

Very simple data preprocessing. We read our text file, split it into sentences, and split each sentence into words.

In [80]:
human_corpus = [s.split() for s in open('human.txt', 'r').read().split(".")]
gpt_corpus = [s.split() for s in open('gpt.txt', 'r').read().split(".")]

# Compute training data

We will look at 5 consecutive sentences at a time. We then calculate the average sentence length and average word length

In [81]:
# X and Y are the actual input and labels for the model
# raw_X will contain all the 5-sentence lists
X, Y, raw_X = [], [], []

def sliding_window(l, n):
    """
    Example for n=2
    ABCDE -> AB, BC, CD, DE
    """
    window = l[:n-1]
    for e in l[n-1:]:
        window.append(e)
        yield window[::]
        window.pop(0)

# Populate raw_X and Y
for sentence in sliding_window(human_corpus, 5):
    raw_X.append(sentence)
    Y.append(0)
for sentence in sliding_window(gpt_corpus, 5):
    raw_X.append(sentence)
    Y.append(1)

def av_element_length(x):
    lengths = [len(y) for y in x]
    return sum(lengths)/len(lengths)

def av_word_length(x):
    words = [word for sentence in x for word in sentence]
    return av_element_length(words)

# Populate X
for x in raw_X:
    sample = []
    sample.append(av_element_length(x))
    sample.append(av_word_length(x))
    X.append(sample)

In [82]:
# Note that this is a data leakage which is fine for now since our model is so dumb
# but should definitely be fixed later
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=67)

# Train model
model = LogisticRegression()
model.fit(X_train, Y_train)
print(f"{model.score(X_test, Y_test)=}")
print(f"{model.coef_=}")

model.score(X_test, Y_test)=0.6923076923076923
model.coef_=array([[ 0.18332849, -1.14067046]])
